# Grade 10 flagship — Air in my city

> **SYNTHETIC** table: do not quote results as real. Run `python tools/fetch_data.py kathmandu-air-quality` for the real series (check the licence first).

Keep the five PPDAC headings.

## P — Problem
*Example: In which months is PM2.5 highest, and how often is a chosen limit exceeded?*

## P — Plan
*Which limit will you use? (Look it up and cite the source.) Which months? How will you treat missing data?*

## D — Data
*Source, licence, date accessed, cleaning log.*

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

air = pd.read_csv(data_file("kathmandu-air-quality", "air_daily_synthetic.csv"), parse_dates=["date"])
air["pm25"] = air["pm25"].replace(-999, np.nan)     # -999 is a sensor error code, not a reading
print(air.shape, "| missing PM2.5:", int(air["pm25"].isna().sum()))

## A — Analysis

In [ ]:
LIMIT = 50   # µg/m³ — PLACEHOLDER: replace with a real guideline value and cite its source
air["month"] = air["date"].dt.month
air["over"] = air["pm25"] > LIMIT
summary = air.groupby("month").agg(mean_pm25=("pm25", "mean"), days_over=("over", "sum"), days_valid=("pm25", "count")).round(1)
summary["share_over_pct"] = (100 * summary["days_over"] / summary["days_valid"]).round(1)
assert (summary["days_valid"] > 0).all()
summary

In [ ]:
summary["share_over_pct"].plot(kind="bar", title="Share of days over the limit, by month (SYNTHETIC)", ylabel="% of valid days")
plt.show()

## C — Conclusion
*Claim · Evidence · Reasoning · Limits (one station? missing days? synthetic?) · AI help used for: ___*